# 🩺 Clinical Symptom Screening Assistant for Nurses

**Author:** Tasneem Alassaf

## Objective
Build a nurse-friendly, safety-aware clinical screening assistant for early patient triage.

## Models
- **Model A:** Logistic Regression (scikit-learn)  
- **Model B:** Flan-T5-base (CPU-friendly)

## Data
Publicly available medical symptom datasets and curated clinical text references.

## Final UI
Simple nurse-focused interface:
symptom input → extracted symptoms → Model A predictions → Model B guidance → escalation notice


# **Install Dependencies**

In [2]:
!pip -q install gradio==4.42.0 pandas scikit-learn kagglehub transformers==4.44.2 torch accelerate
!pip -q install fuzzywuzzy python-Levenshtein # Installing fuzzywuzzy separately due to previous conflict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.23.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 12.0 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 12.0 which is inco

# **Download Datasets (kagglehub)**

In [3]:
import kagglehub
import os

print("Downloading main dataset...")
path1 = kagglehub.dataset_download("itachi9604/disease-symptom-description-dataset")
print("Dataset 1 path:", path1)

print("Downloading secondary dataset...")
path2 = kagglehub.dataset_download("uom190346a/disease-symptoms-and-patient-profile-dataset")
print("Dataset 2 path:", path2)

print("Files in main dataset:", os.listdir(path1))
print("Files in secondary dataset:", os.listdir(path2))

100%|██████████| 30.1k/30.1k [00:00<00:00, 37.5MB/s]

Extracting files...
Dataset 1 path: /root/.cache/kagglehub/datasets/itachi9604/disease-symptom-description-dataset/versions/2


100%|██████████| 3.07k/3.07k [00:00<00:00, 5.73MB/s]

Extracting files...
Dataset 2 path: /root/.cache/kagglehub/datasets/uom190346a/disease-symptoms-and-patient-profile-dataset/versions/2
Files in main dataset: ['symptom_Description.csv', 'symptom_precaution.csv', 'dataset.csv', 'Symptom-severity.csv']
Files in secondary dataset: ['Disease_symptom_and_patient_profile_dataset.csv']


# **Load Data & Create Lookup Maps**

In [4]:

import os
import pandas as pd

# Support both variable names (path1 from your code / path_main from mine)
DATA_ROOT = path1 if "path1" in globals() else path_main

data_file = os.path.join(DATA_ROOT, "dataset.csv")
desc_file = os.path.join(DATA_ROOT, "symptom_Description.csv")
prec_file = os.path.join(DATA_ROOT, "symptom_precaution.csv")

# Load CSVs
df = pd.read_csv(data_file)
desc_df = pd.read_csv(desc_file)
prec_df = pd.read_csv(prec_file)

# Clean disease names to avoid mismatch later
df["Disease"] = df["Disease"].astype(str).str.strip()
desc_df["Disease"] = desc_df["Disease"].astype(str).str.strip()
prec_df["Disease"] = prec_df["Disease"].astype(str).str.strip()

# Build Description map: Disease -> Description
desc_map = dict(
    zip(
        desc_df["Disease"],
        desc_df["Description"].astype(str).str.strip()
    )
)

# Build Precaution map: Disease -> "p1, p2, p3, p4"
prec_cols = [c for c in prec_df.columns if "precaution" in c.lower()]

prec_map = {}
for _, row in prec_df.iterrows():
    d = row["Disease"]
    p_list = []
    for c in prec_cols:
        val = row.get(c)
        if pd.notna(val) and str(val).strip():
            p_list.append(str(val).strip())
    prec_map[d] = ", ".join(p_list)

# Evidence prints for report
n_records = len(df)
n_diseases = df["Disease"].nunique()
print(f"Loaded {n_records} records")
print(f"Unique diseases in dataset: {n_diseases}")
print(f"Descriptions available: {len(desc_map)}")
print(f"Precautions available: {len(prec_map)}")

# Quick sanity checks (helps for debugging + screenshots)
print("\nSample disease:", df['Disease'].iloc[0])
print("Description:", desc_map.get(df['Disease'].iloc[0], "N/A")[:120], "...")
print("Precautions:", prec_map.get(df['Disease'].iloc[0], "N/A"))


Loaded 4920 records
Unique diseases in dataset: 41
Descriptions available: 41
Precautions available: 41

Sample disease: Fungal infection
Description: In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune ...
Precautions: bath twice, use detol or neem in bathing water, keep infected area dry, use clean cloths


# **Train Model A (Logistic Regression)**

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from fuzzywuzzy import process
import re # Import the re module

SEED = 42
np.random.seed(SEED)

# -----------------------------
# A) Dataset structure
# -----------------------------
print("Columns in df:", df.columns.tolist())

target_col = "Disease"
assert target_col in df.columns, f"'{target_col}' column not found!"

X_raw = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].astype(str).str.strip()

# -----------------------------
# B) Build symptom vocabulary from dataset values
#    (works for Symptom_1..Symptom_n style datasets)
# -----------------------------
def normalize_symptom(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace(" ", "_")
    s = re.sub(r"[^a-z0-9_]+", "", s)  # keep simple tokens
    return s

symptom_vocab = sorted(set(
    normalize_symptom(val)
    for col in X_raw.columns
    for val in X_raw[col].dropna().astype(str)
    if normalize_symptom(val) not in ["", "nan"]
))

symptom_index = {s: i for i, s in enumerate(symptom_vocab)}
print(f"Built symptom vocabulary: {len(symptom_vocab)} symptoms")

# -----------------------------
# C) Row -> multi-hot vector
# -----------------------------
def row_to_multihot(row) -> np.ndarray:
    vec = np.zeros(len(symptom_vocab), dtype=np.int8)
    for val in row:
        if pd.isna(val):
            continue
        token = normalize_symptom(val)
        if token in symptom_index:
            vec[symptom_index[token]] = 1
    return vec

X = np.stack([row_to_multihot(r) for _, r in X_raw.iterrows()], axis=0)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)

# -----------------------------
# D) Train / Test split (stratified) + Train LR
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

lr = LogisticRegression(
    max_iter=2500,
    n_jobs=-1,
    class_weight="balanced"   # helps if disease distribution is imbalanced
)
lr.fit(X_train, y_train)

acc = lr.score(X_test, y_test)
print(f"Model A (Logistic Regression) test accuracy: {acc:.4f} ({acc*100:.2f}%)")

# -----------------------------
# E) Free-text symptom extraction (fuzzy >= 88%)
#    This matches the report: "rule-based + fuzzy matching threshold 88%"
# -----------------------------
FUZZY_THRESHOLD = 88

def extract_symptoms_from_text(text: str, max_symptoms: int = 12):
    """
    Extract candidate symptoms from free-text using tokenization + fuzzy matching.
    Returns a list of normalized symptom tokens that exist in symptom_vocab.
    """
    if not text or not str(text).strip():
        return []

    # basic token chunks (kept simple for reproducibility)
    raw = str(text).lower()
    raw = re.sub(r"[^a-z0-9,\s\-_/]+", " ", raw)
    chunks = re.split(r"[,\n;/]+", raw)
    chunks = [c.strip() for c in chunks if c.strip()]

    found = []
    for chunk in chunks:
        # fuzzy match chunk to vocab
        best = process.extractOne(chunk, symptom_vocab)
        if best:
            match, score = best[0], best[1]
            if score >= FUZZY_THRESHOLD and match not in found:
                found.append(match)
        if len(found) >= max_symptoms:
            break

    return found

def symptoms_to_vector(symptoms_list):
    vec = np.zeros(len(symptom_vocab), dtype=np.int8)
    for s in symptoms_list:
        s = normalize_symptom(s)
        if s in symptom_index:
            vec[symptom_index[s]] = 1
    return vec

# -----------------------------
# F) Predict Top-K diseases (for UI + report screenshots)
# -----------------------------
def predict_topk(symptoms_text: str, k: int = 5):
    extracted = extract_symptoms_from_text(symptoms_text)

    vec = symptoms_to_vector(extracted).reshape(1, -1)

    proba = lr.predict_proba(vec)[0]
    top_idx = np.argsort(proba)[::-1][:k]

    results = []
    for idx in top_idx:
        disease = le.inverse_transform([idx])[0]
        conf = float(proba[idx])
        results.append({"disease": disease, "confidence": conf})

    return extracted, results

# Quick sanity test (take screenshot later)
demo_text = "high fever, cough, fatigue"
extracted_demo, preds_demo = predict_topk(demo_text, k=5)

print("\nDemo input:", demo_text)
print("Extracted symptoms:", extracted_demo)
print("Top-5 predictions:")
for r in preds_demo:
    print(f"- {r['disease']}: {r['confidence']:.2f}")

Columns in df: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']
Built symptom vocabulary: 131 symptoms
Model A (Logistic Regression) test accuracy: 1.0000 (100.00%)

Demo input: high fever, cough, fatigue
Extracted symptoms: ['high_fever', 'cough', 'fatigue']
Top-5 predictions:
- Bronchial Asthma: 0.38
- AIDS: 0.09
- Jaundice: 0.04
- Pneumonia: 0.04
- Impetigo: 0.04


In [21]:
# =========================
# FIX: DESC/PREC for Model A labels (Category-level)
# Paste this cell AFTER loading desc_map/prec_map
# =========================

CATEGORY_DESC_PREC = {
    "Respiratory condition": {
        "desc": "Pattern suggests respiratory involvement (e.g., cough, fever, breathing discomfort). This is NOT a diagnosis.",
        "prec": "Check RR and SpO2, assess breathing effort, ask about onset, and monitor for deterioration."
    },
    "General infection": {
        "desc": "Pattern suggests a systemic infection-like presentation (e.g., fever, fatigue, malaise). This is NOT a diagnosis.",
        "prec": "Check vitals, hydration status, mental status; reassess frequently; escalate if worsening."
    },
    "Viral syndrome": {
        "desc": "Pattern suggests a viral-like illness with generalized symptoms. This is NOT a diagnosis.",
        "prec": "Supportive monitoring, track fever trend, assess for red flags, and follow local protocol."
    },
    "Unclear": {
        "desc": "Insufficient or non-specific symptoms to confidently categorize. This is NOT a diagnosis.",
        "prec": "Gather more history, clarify severity/duration, repeat assessment, and consider clinician review."
    },
}

def get_desc_prec(label: str):
    """
    Returns (desc, prec) for:
    1) Exact category labels (Respiratory condition, etc.)
    2) Kaggle disease names via desc_map/prec_map (if available)
    Otherwise returns '—'
    """
    label = (label or "").strip()

    # 1) Category-level fallback (your current labels)
    if label in CATEGORY_DESC_PREC:
        return CATEGORY_DESC_PREC[label]["desc"], CATEGORY_DESC_PREC[label]["prec"]

    # 2) Kaggle disease name (if your model outputs real diseases later)
    desc = (desc_map.get(label) or "").strip() if "desc_map" in globals() else ""
    prec = (prec_map.get(label) or "").strip() if "prec_map" in globals() else ""

    if desc or prec:
        return (desc if desc else "—"), (prec if prec else "—")

    return "—", "—"


# **5.Load Model B (Flan-T5-base – CPU friendly)**

In [6]:


from transformers import pipeline

pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1  # CPU
)

print("Model B loaded successfully (Flan-T5-base, CPU)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Model B loaded successfully (Flan-T5-base, CPU)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# **6.Core Functions (Extraction, Prompt, Escalation)**

In [7]:
!pip install -q fuzzywuzzy python-Levenshtein


import re, os
import numpy as np
import pandas as pd
from fuzzywuzzy import process
from datetime import datetime

DISCLAIMER = (
    "DISCLAIMER: This is NOT a medical diagnosis. "
    "This tool is for educational decision-support only. "
    "Always consult a licensed healthcare professional."
)

FUZZY_THRESHOLD = 88

# Red-flag / escalation keywords
RED_FLAGS_HIGH = [
    "chest pain", "difficulty breathing", "shortness of breath", "unconscious",
    "seizure", "confusion"
]
MENTAL_HEALTH_FLAGS = ["suicide", "self harm", "kill myself", "hurt myself"]

# Optional: temperature patterns (simple)
TEMP_PATTERN = re.compile(r"\b(39\.5|40|41)\b")

# -----------------------------
# A) Symptom extraction
# -----------------------------
# Precompute vocab as "space form" for faster matching
symptom_vocab_space = [s.replace("_", " ") for s in symptom_vocab]

def extract_symptoms(text: str, max_symptoms: int = 12):
    """
    Rule-based: exact phrase contains
    Fuzzy-based: chunk-level fuzzy match (>= 88) to reduce noise
    """
    if not text or not str(text).strip():
        return []

    t = str(text).lower()
    t = re.sub(r"[^a-z0-9,\s\-_/]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    found = []

    # 1) Exact phrase match (strong signal)
    for s in symptom_vocab:
        phrase = s.replace("_", " ")
        if phrase in t:
            found.append(s)

    # 2) Chunk fuzzy match (safer than single-word fuzzy)
    chunks = re.split(r"[,\n;/]+", t)
    chunks = [c.strip() for c in chunks if c.strip()]

    for chunk in chunks:
        best = process.extractOne(chunk, symptom_vocab_space)
        if best:
            match_space, score = best
            if score >= FUZZY_THRESHOLD:
                found.append(match_space.replace(" ", "_"))
        if len(set(found)) >= max_symptoms:
            break

    return sorted(set(found))

# -----------------------------
# B) Model A prediction helpers
# -----------------------------
def symptoms_to_vector(symptoms_list):
    vec = np.zeros(len(symptom_vocab), dtype=np.int8)
    for s in symptoms_list:
        if s in symptom_index:
            vec[symptom_index[s]] = 1
    return vec.reshape(1, -1)

def predict_topk_from_symptoms(symptoms_list, k=5):
    vec = symptoms_to_vector(symptoms_list)
    proba = lr.predict_proba(vec)[0]
    top_idx = np.argsort(proba)[::-1][:k]
    results = []
    for idx in top_idx:
        disease = le.inverse_transform([idx])[0]
        conf = float(proba[idx])
        results.append((disease, conf))
    return results

def format_model_a(top_pairs, min_conf=0.05):
    lines = []
    for rank, (d, p) in enumerate(top_pairs, 1):
        if p < min_conf:
            continue
        desc = (desc_map.get(d, "No description available.") or "")[:220]
        prec = (prec_map.get(d, "No precautions available.") or "")[:220]
        lines.append(
            f"{rank}. {d} (Conf: {p:.1%})\n"
            f"   Desc: {desc}\n"
            f"   Prec: {prec}\n"
        )
    return "\n".join(lines).strip() if lines else "(no confident predictions)"

# -----------------------------
# C) Model B prompt (STRICT 4 bullets)
# -----------------------------
PROMPT_TEMPLATE = """
You are a nurse triage assistant. Be concise, calm, and safety-focused.

Patient symptoms (extracted):
{symptoms}

Context (medical reference):
{context}

Return EXACTLY 4 bullet points with these headings:
- Main concern:
- Nurse actions:
- When to escalate:
- Next step:

Rules:
- Do NOT diagnose or name diseases.
- Do NOT recommend medications or dosages.
- Do NOT repeat sentences.
- Keep each bullet short and practical.

End with:
{disclaimer}
"""

def build_grounding_context(top_pairs):
    """
    Use top-1 disease ONLY for grounding text (not as a diagnosis).
    """
    if not top_pairs:
        return "No reference context available."

    top_disease = top_pairs[0][0]
    parts = []
    if top_disease in desc_map:
        parts.append(desc_map[top_disease])
    if top_disease in prec_map:
        parts.append("Precautions: " + prec_map[top_disease])

    ctx = " ".join([p for p in parts if p]).strip()
    return (ctx[:800] if ctx else "No reference context available.")

def generate_model_b(symptoms_list, top_pairs):
    symptoms_str = ", ".join(symptoms_list) if symptoms_list else "none"
    context_text = build_grounding_context(top_pairs)

    prompt = PROMPT_TEMPLATE.format(
        symptoms=symptoms_str,
        context=context_text,
        disclaimer=DISCLAIMER
    )

    out = pipe(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4
    )[0]["generated_text"].strip()

    return out

# -----------------------------
# D) Tiered escalation (Low/Med/High) + suppression
# -----------------------------
def escalation_level(user_text, symptoms_list, top_pairs):
    text = (user_text or "").lower()

    # High severity: red flags or mental health flags
    if any(k in text for k in MENTAL_HEALTH_FLAGS):
        return 3, "Possible mental health emergency (self-harm risk)."
    if any(k in text for k in RED_FLAGS_HIGH) or TEMP_PATTERN.search(text):
        return 3, "Potential emergency symptoms detected (red flags)."

    # Low/Medium based on confidence + symptom count + ambiguity
    if not top_pairs:
        return 2, "No reliable predictions available."

    top1 = top_pairs[0][1]
    top3 = [p for _, p in top_pairs[:3]]

    if top1 < 0.35 or len(symptoms_list) < 3:
        return 1, "Low confidence or too few symptoms."

    if all(p < 0.20 for p in top3):
        return 2, "Ambiguous symptoms (multiple low-confidence candidates)."

    return 0, "No immediate escalation."

def render_escalation_note(level, reason):
    if level == 0:
        return f"✅ OK: {reason}\n\n{DISCLAIMER}"
    if level == 1:
        return f"⚠️ LEVEL 1 (Low): {reason}\nAction: Clarify symptoms, check vitals, monitor.\n\n{DISCLAIMER}"
    if level == 2:
        return f"🟠 LEVEL 2 (Medium): {reason}\nAction: Strong recommendation for clinician review.\n\n{DISCLAIMER}"
    return f"🚨 LEVEL 3 (High): {reason}\nAction: Escalate to physician/ER immediately.\n\n{DISCLAIMER}"

# -----------------------------
# E) Logging (Evidence for report)
# -----------------------------
FEEDBACK_LOG = "feedback_log.csv"
ESCALATION_LOG = "escalations_log.csv"

def log_escalation(user_text, symptoms_list, top_pairs, level, reason):
    row = {
        "timestamp": datetime.utcnow().isoformat(),
        "input_text": user_text,
        "extracted_symptoms": ", ".join(symptoms_list),
        "top1_disease": top_pairs[0][0] if top_pairs else "",
        "top1_conf": float(top_pairs[0][1]) if top_pairs else "",
        "level": level,
        "reason": reason
    }
    df_log = pd.DataFrame([row])
    if os.path.exists(ESCALATION_LOG):
        df_log.to_csv(ESCALATION_LOG, mode="a", header=False, index=False)
    else:
        df_log.to_csv(ESCALATION_LOG, index=False)

# -----------------------------
# F) Main function used by UI
# -----------------------------
def run_ui(user_text):
    symptoms = extract_symptoms(user_text)
    symptoms_str = ", ".join(symptoms) if symptoms else "none"

    # If nothing extracted -> clarify (Level 1 style)
    if not symptoms:
        note = render_escalation_note(1, "No symptoms detected from input.")
        # no models shown
        return symptoms_str, "(no predictions)", "(no guidance)", note

    # Model A
    top_pairs = predict_topk_from_symptoms(symptoms, k=5)
    a_out = format_model_a(top_pairs, min_conf=0.05)

    # Escalation
    level, reason = escalation_level(user_text, symptoms, top_pairs)
    note = render_escalation_note(level, reason)

    # Model B (suppressed on Level 3)
    if level == 3:
        b_out = "(suppressed due to high-risk escalation)"
    else:
        b_out = generate_model_b(symptoms, top_pairs)

    # Log escalation (evidence)
    log_escalation(user_text, symptoms, top_pairs, level, reason)

    return symptoms_str, a_out, b_out, note


# **RAG**

In [8]:
!pip -q install -U pypdf sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.3 MB/s eta 0:00:00


In [9]:
from google.colab import files
import os, re, shutil
from pypdf import PdfReader

# ---------- Upload PDFs ----------
uploaded = files.upload()

os.makedirs("data", exist_ok=True)
for fn in list(uploaded.keys()):
    if fn.lower().endswith(".pdf"):
        shutil.move(fn, os.path.join("data", fn))

print("PDFs in data/:", os.listdir("data"))

# ---------- Strong medical cleaning ----------
def clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    # 1) Fix hyphenated splits across spaces/newlines: feel- ings -> feelings, ill- ness -> illness
    text = re.sub(r"(\w)\s*-\s+(\w)", r"\1\2", text)

    # 2) Remove common PDF noise
    text = re.sub(r"Page\s*\d+\s*of\s*\d+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+,", ",", text)         # "antibiotics ," -> "antibiotics,"
    text = re.sub(r"\s+\.", ".", text)        # "word ." -> "word."
    text = re.sub(r"\s+;", ";", text)

    # 3) Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 4) Drop very short fragments that usually come from broken page headers/footers observed in books
    if len(text) < 120:
        return ""
    return text

def chunk_text(text: str, chunk_size=900, overlap=150):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(n, start + chunk_size)
        ch = text[start:end].strip()
        if len(ch) >= 200:
            chunks.append(ch)
        start += (chunk_size - overlap)
    return chunks

# ---------- Load PDFs + build chunks ----------
final_chunks = []  # list of dicts {page_content, source, page}

for fn in os.listdir("data"):
    if not fn.lower().endswith(".pdf"):
        continue

    path = os.path.join("data", fn)
    reader = PdfReader(path)

    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        cleaned = clean_pdf_text(raw)
        if not cleaned:
            continue

        for ch in chunk_text(cleaned, chunk_size=900, overlap=150):
            final_chunks.append({
                "page_content": ch,
                "source": fn,
                "page": i
            })

print("Total chunks:", len(final_chunks))

# Quick check: print one cleaned chunk
if final_chunks:
    print("\n--- SAMPLE CLEAN CHUNK ---")
    print(final_chunks[0]["page_content"][:600])
    print(f"\n(source: {final_chunks[0]['source']}, page: {final_chunks[0]['page']})")


Saving The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf to The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf
PDFs in data/: ['The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf']
Total chunks: 4318

--- SAMPLE CLEAN CHUNK ---
STAFF Jacqueline L. Longe, Project Editor Deirdre S. Blanchfield, Associate Editor Christine B. Jeryan, Managing Editor Donna Olendorf, Senior Editor Stacey Blachford, Associate Editor Kate Kretschmann, Melissa C. McDade, Ryan Thomason, Assistant Editors Mark Springer, Technical Specialist Andrea Lopeman, Programmer/Analyst Barbara J. Yarrow,Manager, Imaging and Multimedia Content Robyn V. Young,Project Manager, Imaging and Multimedia Content Dean Dauphinais, Senior Editor, Imaging and Multimedia Content Kelly A. Quin, Editor, Imaging and Multimedia Content Leitha Etheridge-Sims, Mary K. Grime

(source: The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf, page: 3)


In [10]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

if len(final_chunks) == 0:
    index = None
    embeddings = None
    print("No chunks available. RAG disabled.")
else:
    texts = [c["page_content"] for c in final_chunks]
    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    print("FAISS index ready:", embeddings.shape)

def retrieve_context(query: str, k: int = 4, max_chars: int = 1200) -> str:
    if index is None or embeddings is None or len(final_chunks) == 0:
        return ""

    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    distances, idxs = index.search(q, k)

    pieces = []
    for idx in idxs[0]:
        ch = final_chunks[int(idx)]
        src = f"(source: {ch['source']}, page: {ch['page']})"
        pieces.append(ch["page_content"] + "\n" + src)

    ctx = "\n\n".join(pieces).strip()
    return ctx[:max_chars]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

FAISS index ready: (4318, 384)


In [11]:
print(retrieve_context("fatigue drowsiness antihistamines antibiotics", k=2))


In addition, mental disorders such as depression can also cause fatigue. A number of medications, including antihistamines, antibiotics, and blood pressure medications, may cause drowsiness as a side-effect. Individuals already suffering from fatigue who are prescribed one of these medications may wish to check with their healthcare provider about alternative treatments. Extreme fatigue which persists, unabated, for at least six months, is not the result of a diagnosed disease or illness, and is characterized by flu-like symptoms such as swollen lymph nodes, sore throat, and muscle weakness and/or pain may indicate a diagnosis of chronic fatigue syndrome. Chronic fatigue syndrome (sometimes called chronic fatigue immune deficiency syndrome), is a debilitating illness that causes overwhelming exhaustion and a constellation of neurological and immunological symptoms. Between 1.5 and 2 mill
(source: The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf, page: 681)

cells lining the nasal passages.

In [12]:
print(retrieve_context("fever cough fatigue", k=2)[:800])


tickle in the throat, runny nose, and sneezing. The initial discharge from the nose is clear and thin. Later it changes to a thick yellow or greenish discharge. Most adults do not develop a fever when they catch a cold. Young children may develop a low fever of up to 102°F (38.9°C). In addition to a runny nose and fever, signs of a cold include coughing, sneezing, nasal congestion, headache, muscle ache, chills, sore throat, hoarseness, watery eyes, tiredness, and lack of appetite. The cough that accompanies a cold is usually intermittent and dry. Most people begin to feel better four to five days after their cold symptoms become noticeable. All symptoms are generally gone within ten days, except for a dry cough that may linger for up to three weeks. Colds make people more susceptible to b


# **7.Final Nurse-Friendly UI**

In [23]:
import gradio as gr
import numpy as np
import re

DISCLAIMER = (
    "Educational prototype only. Not a medical diagnosis. "
    "Always consult a licensed healthcare professional."
)

# -----------------------------
# DESC/PREC getter
# -----------------------------
def _get_desc_prec(disease: str):
    return get_desc_prec(disease)  # <-- uses your compact category map / kaggle map

# -----------------------------
# Model A formatter (clean cards)
# -----------------------------
def _format_model_a(top_pairs):
    blocks = []
    for rank, (disease, prob) in enumerate(top_pairs, 1):
        desc, prec = _get_desc_prec(disease)
        desc = re.sub(r"\s+", " ", (desc or "")).strip()
        prec = re.sub(r"\s+", " ", (prec or "")).strip()

        # keep short in UI (professional)
        if len(desc) > 160: desc = desc[:160].rstrip() + "…"
        if len(prec) > 140: prec = prec[:140].rstrip() + "…"

        color = "#b71c1c" if rank == 1 else "#1b5e20"
        blocks.append(f"""
        <div style="border:1px solid #e5e7eb;border-radius:14px;padding:12px 14px;margin-bottom:10px;">
          <div style="display:flex;justify-content:space-between;align-items:center;">
            <div style="font-weight:900;color:{color};font-size:15px;">{rank}. {disease}</div>
            <div style="font-weight:800;color:{color};">Confidence: {prob:.2f}</div>
          </div>
          <div style="margin-top:8px;"><b>Description:</b> {desc if desc else "—"}</div>
          <div style="margin-top:4px;"><b>Precautions:</b> {prec if prec else "—"}</div>
        </div>
        """)

    return "\n".join(blocks) if blocks else "<i>No predictions.</i>"

# -----------------------------
# Safety note (simple + clear)
# -----------------------------
def _safety_note(user_text, symptoms, top_prob):
    t = (user_text or "").lower()
    red_flags = ["chest pain", "difficulty breathing", "shortness of breath", "seizure", "unconscious", "self-harm", "suicide"]

    if any(r in t for r in red_flags):
        return "ESCALATION: Red-flag symptoms detected → escalate immediately per protocol.\n\n" + DISCLAIMER

    msgs = []
    if top_prob < 0.35:
        msgs.append("Low confidence → gather more symptoms / clinician review.")
    if len(symptoms) < 3:
        msgs.append("Few symptoms → clarify onset, duration, severity, and associated factors.")

    if msgs:
        return "ESCALATION:\n- " + "\n- ".join(msgs) + "\n\n" + DISCLAIMER

    return "No immediate escalation. Monitor and reassess.\n\n" + DISCLAIMER

# -----------------------------
# Model B prompt (NO '4 lines' mention)
# -----------------------------
def _prompt_model_b(symptoms_str, note):
    return f"""
You are a SAFE nurse screening assistant (screening support only).
Do NOT diagnose. Do NOT prescribe medications or dosages.
Be concise, practical, and nurse-focused.

Write a short structured response with these labels:
Main concern:
Nurse actions:
When to escalate:
Next step:

Symptoms: {symptoms_str}

Safety note: {note}
""".strip()

# -----------------------------
# Main run
# -----------------------------
def run_ui(user_text):
    # guard: required objects exist
    required = ["extract_symptoms", "symptom_vocab", "symptom_index", "lr", "le", "pipe"]
    missing = [x for x in required if x not in globals()]
    if missing:
        msg = "Missing objects (run previous cells): " + ", ".join(missing)
        return "none", "<i>(Model A unavailable)</i>", msg, msg

    symptoms = extract_symptoms(user_text) or []
    symptoms_str = ", ".join(symptoms) if symptoms else "none detected"

    if not symptoms:
        note = "Few / no matched symptoms → ask for more details."
        b_out = (
            "Main concern: Insufficient symptom details for safe screening.\n"
            "Nurse actions: Ask about onset/duration/severity; check vitals; document.\n"
            "When to escalate: Any red flags or unstable vitals.\n"
            "Next step: Reassess after collecting more information.\n\n"
            + DISCLAIMER
        )
        return symptoms_str, "<i>(no matching symptoms)</i>", b_out, _safety_note(user_text, symptoms, 0.0)

    # vectorize for Model A
    vec = np.zeros(len(symptom_vocab), dtype=float)
    for s in symptoms:
        if s in symptom_index:
            vec[symptom_index[s]] = 1.0
    vec = vec.reshape(1, -1)

    probs = lr.predict_proba(vec)[0]
    top_idx = np.argsort(probs)[::-1][:5]
    top_pairs = [(le.inverse_transform([i])[0], float(probs[i])) for i in top_idx]

    top_prob = top_pairs[0][1] if top_pairs else 0.0
    note = _safety_note(user_text, symptoms, top_prob)

    # Model A HTML
    a_html = _format_model_a(top_pairs)

    # Model B
    prompt = _prompt_model_b(symptoms_str, note)
    b = pipe(
        prompt,
        max_new_tokens=160,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4
    )[0]["generated_text"].strip()

    if "DISCLAIMER" not in b.upper():
        b += "\n\n" + DISCLAIMER

    return symptoms_str, a_html, b, note

# -----------------------------
# UI (clean professional)
# -----------------------------
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"),
    title="Clinical Symptom Screening Assistant"
) as demo:

    gr.Markdown("# 🩺 Clinical Symptom Screening Assistant for Nurses")
    gr.Markdown("**Author:** Tasneem Alassaf")

    gr.Markdown(
        f"<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#b71c1c;"
        f"padding:12px;border-radius:12px;font-weight:700;margin-bottom:12px;'>"
        f"{DISCLAIMER}</div>"
    )

    inp = gr.Textbox(
        lines=6,
        label="Patient Symptoms (free-text)",
        placeholder="e.g., high fever for 4 days, dry cough, severe fatigue"
    )

    with gr.Row():
        btn = gr.Button("Run Screening", variant="primary")
        clear = gr.Button("Clear")

    with gr.Row(equal_height=True):
        with gr.Column():
            out_sym = gr.Textbox(label="Extracted Symptoms", lines=2)
            out_a = gr.HTML(label="Model A — Predictions (with confidence)")
        with gr.Column():
            out_b = gr.Textbox(label="Model B — Nurse Guidance", lines=10, show_copy_button=True)
            out_note = gr.Textbox(label="Safety & Escalation", lines=6, interactive=False)

    btn.click(run_ui, inputs=[inp], outputs=[out_sym, out_a, out_b, out_note])
    clear.click(lambda: ("", "", "", ""), outputs=[inp, out_sym, out_a, out_b])

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.42.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on public URL: https://0b599d93f13d350eb0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7ea4546a9930542556.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://4661c34705f4df26a2.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://b33bebbbb9704411b0.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://0b599d93f13d350eb0.gradio.live
